# D04 — Reference Single-Cell Datasets

Downloads the single-cell RNA-seq reference datasets used for in silico
perturbation experiments (Notebooks 05, 06).

**Outputs (in `data/external/`):**
- `pbmc3k_processed.h5ad` — PBMC3k processed dataset (ScanPy built-in)
- `pbmc3k_raw.h5ad` — PBMC3k raw counts

**Prerequisites:** `scanpy`, `anndata`, `requests`

**Note:** The perturbation experiments require running live Geneformer and
scGPT models on these datasets, which is computationally intensive (P06, P07).
This notebook documents the reference data acquisition only.

## Section 1: PBMC3k (Peripheral Blood Mononuclear Cells)

The 10x Genomics PBMC3k dataset is a standard benchmark for single-cell analysis, containing ~2,700 cells across 8 immune cell types. We use this dataset for in silico perturbation experiments to measure single-gene deletion sensitivity and mask-in-place controls.

In [1]:
import scanpy as sc
import anndata
from pathlib import Path

# anndata >= 0.11 requires explicit opt-in for nullable string arrays
try:
    anndata.settings.allow_write_nullable_strings = True
except AttributeError:
    pass  # older anndata versions don't have this setting

DATA_DIR = Path('data/external')
DATA_DIR.mkdir(parents=True, exist_ok=True)

PBMC_PROC = DATA_DIR / 'pbmc3k_processed.h5ad'
PBMC_RAW = DATA_DIR / 'pbmc3k_raw.h5ad'

print('PBMC3k dataset')
print('=' * 50)
print()
print('Note: Downstream P-series and figure notebooks use pre-computed CSVs from')
print('the analysis pipeline, not these .h5ad files directly. These reference')
print('datasets are provided for full reproducibility of model-dependent')
print('experiments (P06–P09).')
print()

if PBMC_PROC.exists():
    try:
        adata = anndata.read_h5ad(PBMC_PROC)
        print(f'Processed PBMC3k already cached: {adata.shape}')
    except Exception as e:
        print(f'Cached file exists but cannot be read ({e})')
        print('Re-downloading...')
        adata_proc = sc.datasets.pbmc3k_processed()
        adata_proc.write_h5ad(PBMC_PROC)
        print(f'Saved processed: {adata_proc.shape} to {PBMC_PROC}')
else:
    print('Downloading PBMC3k processed dataset via ScanPy...')
    adata_proc = sc.datasets.pbmc3k_processed()
    adata_proc.write_h5ad(PBMC_PROC)
    print(f'Saved processed: {adata_proc.shape} to {PBMC_PROC}')

if PBMC_RAW.exists():
    try:
        adata_raw = anndata.read_h5ad(PBMC_RAW)
        print(f'Raw PBMC3k already cached: {adata_raw.shape}')
    except Exception as e:
        print(f'Cached file exists but cannot be read ({e})')
        print('Re-downloading...')
        adata_raw = sc.datasets.pbmc3k()
        sc.pp.filter_cells(adata_raw, min_genes=200)
        sc.pp.filter_genes(adata_raw, min_cells=3)
        adata_raw.write_h5ad(PBMC_RAW)
        print(f'Saved raw (QC-filtered): {adata_raw.shape} to {PBMC_RAW}')
else:
    print('Downloading PBMC3k raw dataset via ScanPy...')
    adata_raw = sc.datasets.pbmc3k()
    sc.pp.filter_cells(adata_raw, min_genes=200)
    sc.pp.filter_genes(adata_raw, min_cells=3)
    adata_raw.write_h5ad(PBMC_RAW)
    print(f'Saved raw (QC-filtered): {adata_raw.shape} to {PBMC_RAW}')

PBMC3k dataset

Note: Downstream P-series and figure notebooks use pre-computed CSVs from
the analysis pipeline, not these .h5ad files directly. These reference
datasets are provided for full reproducibility of model-dependent
experiments (P06–P09).

Processed PBMC3k already cached: (2638, 1838)
Raw PBMC3k already cached: (2700, 13714)


In [2]:
import anndata
adata = anndata.read_h5ad(PBMC_PROC)
print(f'Cells: {adata.n_obs:,}')
print(f'Genes: {adata.n_vars:,}')
print(f'\nCell types (louvain):')
print(adata.obs['louvain'].value_counts().to_string())

Cells: 2,638
Genes: 1,838

Cell types (louvain):
louvain
CD4 T cells          1144
CD14+ Monocytes       480
B cells               342
CD8 T cells           316
NK cells              154
FCGR3A+ Monocytes     150
Dendritic cells        37
Megakaryocytes         15


In [3]:
ALLEN_MEDIANS = DATA_DIR / 'allen_cluster_medians.csv'

if ALLEN_MEDIANS.exists():
    import pandas as pd
    medians = pd.read_csv(ALLEN_MEDIANS, nrows=5)
    full = pd.read_csv(ALLEN_MEDIANS)
    print(f'\nAllen cluster medians: {full.shape}')
    print(f'Clusters: {full.shape[0]}, Genes: {full.shape[1] - 1}')
else:
    print(f'Allen cluster medians not found at {ALLEN_MEDIANS}')

Allen cluster medians not found at data/external/allen_cluster_medians.csv


## Section 3: Pre-computed Experiment Results

The following CSV files contain results from running the live foundation models on the reference datasets above. These experiments are computationally intensive (requiring GPU inference) and are provided pre-computed. The original research notebooks that generated them are preserved in the project root directory (gf_03*, gf_05*, scgpt_03*, etc.).

In [4]:
import pandas as pd
from pathlib import Path

experiment_files = {
    'Perturbation sensitivity (GF)': 'data/perturbation_sensitivity.csv',
    'Perturbation sensitivity (scGPT)': 'data/scgpt_perturbation_sensitivity.csv',
    'scGPT ablation comparison': 'data/scgpt_perturbation_ablations.csv',
    'Mask vs delete control': 'data/mask_perturbation_comparison.csv',
    'Real PBMC validation': 'data/perturbation_real_pbmc.csv',
    'Synthetic vs real comparison': 'data/perturbation_synthetic_vs_real.csv',
}

print('Pre-computed experiment results')
print('=' * 60)
for name, path in experiment_files.items():
    p = Path(path)
    if p.exists():
        df = pd.read_csv(p, nrows=0)
        size = p.stat().st_size / 1024
        unit = 'KB'
        if size > 1024:
            size /= 1024
            unit = 'MB'
        print(f'  ✓ {name:40s}  {size:.1f} {unit}  ({len(df.columns)} cols)')
    else:
        print(f'  ✗ {name:40s}  MISSING')

Pre-computed experiment results
  ✓ Perturbation sensitivity (GF)             17.7 KB  (13 cols)
  ✓ Perturbation sensitivity (scGPT)          3.7 KB  (10 cols)
  ✓ scGPT ablation comparison                 5.8 KB  (14 cols)
  ✓ Mask vs delete control                    12.8 KB  (10 cols)
  ✓ Real PBMC validation                      2.6 KB  (8 cols)
  ✓ Synthetic vs real comparison              3.1 KB  (8 cols)


## Section 4: Verification Summary

In [5]:
print('Verification Summary')
print('=' * 60)
print()
print('Expected outputs in data/external/')
print()

expected_files = {
    'pbmc3k_processed.h5ad': 'PBMC3k processed dataset',
    'pbmc3k_raw.h5ad': 'PBMC3k raw counts (QC-filtered)',
    'allen_cluster_medians.csv': 'Allen cluster median expressions',
}

for filename, description in expected_files.items():
    filepath = DATA_DIR / filename
    if filepath.exists():
        size = filepath.stat().st_size / 1024
        unit = 'KB'
        if size > 1024:
            size /= 1024
            unit = 'MB'
        print(f'  ✓ {filename:30s}  {size:>6.1f} {unit}  — {description}')
    else:
        print(f'  ✗ {filename:30s}  MISSING — {description}')

Verification Summary

Expected outputs in data/external/

  ✓ pbmc3k_processed.h5ad             38.3 MB  — PBMC3k processed dataset
  ✓ pbmc3k_raw.h5ad                   18.9 MB  — PBMC3k raw counts (QC-filtered)
  ✗ allen_cluster_medians.csv       MISSING — Allen cluster median expressions
